# 01 · Preparación e integración de datos

Primero leemos las fuentes oficiales, tratamos espacios como nulos y preservamos los códigos como texto; después seleccionamos 2025 y la cohorte de 6–17 años, unimos los módulos de persona y hogar, agregamos ESCALE por distrito y guardamos la base analítica. Esta es la entrada para los análisis posteriores.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró pipeline.py.")
RAW_ENAHO = ROOT / "data" / "raw" / "enaho"
RAW_ESCALE = ROOT / "data" / "raw" / "minedu"
inputs = [RAW_ENAHO / name for name in [
    "enaho01-2021-2025-100-panel.csv", "enaho01-2021-2025-200-panel.csv",
    "enaho01a-2021-2025-300-panel.csv", "sumaria-2021-2025-panel.csv"
]] + [RAW_ESCALE / "Padron.dbf"]
missing = [path for path in inputs if not path.is_file()]
if missing:
    raise FileNotFoundError("Faltan archivos crudos:\n" + "\n".join(map(str, missing)))
print(f"Fuentes disponibles: {len(inputs)}")

## Ejecutar integración

El script detiene el proceso si faltan columnas o encuentra llaves duplicadas que puedan multiplicar filas.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "src" / "05_integrar_enaho_escale.py")], cwd=ROOT, check=True)

In [ ]:
PROCESSED = ROOT / "data" / "processed"
dataset_path = PROCESSED / "dataset_enaho_escale_2025.parquet"
report_path = PROCESSED / "reporte_integracion_2025.csv"
dictionary_path = PROCESSED / "diccionario_variables_2025.csv"
for path in [dataset_path, report_path, dictionary_path]:
    if not path.is_file():
        raise FileNotFoundError(f"No se generó {path}")
data = pd.read_parquet(dataset_path)
print(f"Base: {len(data):,} filas × {data.shape[1]} columnas")
display(data[["edad", "UBIGEO_25", "CONGLOME_25", "exclusion_digital"]].head())
display(pd.read_csv(report_path))
display(pd.read_csv(dictionary_path))